# Kaggriculture — shop-router screen, stage 1b (tie-break)

Builds the Rust engine for this Linux image, then plays the 228 shortlisted tapes against
the same fixed opponent tape over 30 fresh seeds and records the shops revealed at steps
72 and 144.

**Required notebook settings:** Internet **ON** (the Rust toolchain is downloaded),
Accelerator **None** — the engine is a sequential CPU simulation.

**Run 3 (2026-08-31, stage 1b):** stage 1 finished 16,620 games but could not produce the
40-tape shortlist stage 2 consumes — 12 games per candidate makes point rate a k/12 grid,
and 211 candidates tie at 0.8333 for 23 places. This run re-measures the 228 candidates at
>= 0.8333 on seeds 14150001-14150030 (13,680 games) and ranks on those alone. The runner
now uses `step_typed`, the transport switch scheduled for after stage 1.


In [ ]:
import base64, glob, json, os, re, shutil, subprocess, sys, time

INPUT = glob.glob('/kaggle/input/*/tape_pool')[0].rsplit('/', 1)[0]
POOL  = glob.glob(f'{INPUT}/tape_pool/tape_pool_20260830')[0]
TAPES = f'{POOL}/tapes'
BUILD = '/tmp/build'

# Kaggle auto-extracted the upload, and the macOS tar left AppleDouble "._" siblings.
tape_files = sorted(p for p in glob.glob(f'{TAPES}/*.json')
                    if not os.path.basename(p).startswith('._'))
print(f'input  {INPUT}')
print(f'tapes  {len(tape_files)}')
print(f'cores  {os.cpu_count()}')

shutil.rmtree(BUILD, ignore_errors=True)
shutil.copytree(f'{INPUT}/engine_src', BUILD,
                ignore=shutil.ignore_patterns('._*', '__pycache__'))
# The upload carries the macOS build of the extension; it must not reach the Linux wheel.
for stale in glob.glob(f'{BUILD}/bindings/python/kaggriculture_rs/*.so'):
    os.remove(stale)
    print('removed stale', os.path.basename(stale))
print('build tree ready')

In [ ]:
t0 = time.time()
subprocess.run('curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal '
               '--default-toolchain stable', shell=True, check=True)
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + os.pathsep + os.environ['PATH']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'maturin==1.14.1'], check=True)
print(f'toolchain ready in {time.time() - t0:.0f}s')

In [ ]:
t0 = time.time()
proc = subprocess.run(['maturin', 'build', '--release', '-o', '/kaggle/working/wheels'],
                      cwd=f'{BUILD}/bindings/python', capture_output=True, text=True)
print(proc.stdout[-1500:]); print(proc.stderr[-3000:])
proc.check_returncode()
wheel = glob.glob('/kaggle/working/wheels/*.whl')[0]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', wheel],
               check=True)
print(f'built and installed {os.path.basename(wheel)} in {time.time() - t0:.0f}s')

In [ ]:
from kaggriculture_rs import Engine
probe = Engine({'episodeSteps': 720, 'seed': 1, 'weedSpawnChance': 0.005})
print('engine ok, shops at step 0:', probe.observation(0)['town'].get('unlocked_shops'))

In [ ]:
# Runner is embedded so the notebook is self-contained and we do not re-upload a 158 MB
# source dataset for a one-line change. Patched 2026-08-30: market orders with
# quantity <= 0 are dropped in clean(), because the fail-closed executor rejects them
# (KG-SCHEMA-INVALID) while the Kaggle env treats them as a silent no-op.
# Patched 2026-08-31: the game loop calls step_typed() instead of step(), the transport
# switch the pre-registration schedules for after stage 1. Verified locally on the three
# stage-1b seeds x both seats: identical rows, 1.31x faster.
RUNNER_B64 = (
    'IiIiU2VsZi1jb250YWluZWQgS2FnZ2xlIGJ1aWxkLiBQbGF5IGV2ZXJ5IGNhbmRpZGF0ZSB0YXBlIGFnYWluc3QgZml4ZWQgb3Bwb25lbnQgdGFwZXMg'
    'YW5kIHJlY29yZCB0aGUgcmV2ZWFsZWQgc2hvcHMuCgpFbWl0cyBvbmUgcm93IHBlciAoY2FuZGlkYXRlLCBvcHBvbmVudCwgc2VlZCwgc2VhdCkgd2l0'
    'aCB0aGUgbWFyZ2luIGFuZCB0aGUgZmlyc3QgdHdvCnNob3BzIHJldmVhbGVkIGluIHRoYXQgZ2FtZSwgd2hpY2ggaXMgZXZlcnl0aGluZyB0aGUgVjk0'
    'IGdhdGVzIG5lZWQuIFNlZQpgYW5hbHlzaXMvdjk0X3Nob3Bfcm91dGVyXzIwMjYwODMwL1BSRVJFR0lTVFJBVElPTi5tZGAuCiIiIgoKZnJvbSBfX2Z1'
    'dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIGNvbmN1cnJlbnQuZnV0dXJl'
    'cyBpbXBvcnQgUHJvY2Vzc1Bvb2xFeGVjdXRvcgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKZnJvbSBrYWdn'
    'cmljdWx0dXJlX3JzIGltcG9ydCBFbmdpbmUKCk1BUktFVF9PUFMgPSAoIkJVWV9TRUVEIiwgIkJVWV9QUk9EVUNUIiwgIkJVWV9BTklNQUwiLCAiU0VM'
    'TCIsICJISVJFIiwgIkJVWV9MQU5EIikKTk9fQVJHX09QUyA9ICgiTk9SVEgiLCAiU09VVEgiLCAiRUFTVCIsICJXRVNUIiwgIlBBU1MiLCAiRFJPUCIs'
    'ICJXQVRFUiIsICJIQVJWRVNUIiwKICAgICAgICAgICAgICAiRkVSVElMSVpFIiwgIkRJRyIsICJCVUlMRF9DT09QIiwgIkJVSUxEX1BBU1RVUkUiLCAi'
    'RkVFRCIsICJDQVJFIiwKICAgICAgICAgICAgICAiQ09MTEVDVF9GRVJUSUxJWkVSIikKCgpkZWYgbm9ybWFsaXplX2xlZ2FjeV9hY3Rpb24oYWN0aW9u'
    'KToKICAgICIiIkNhbm9uaWNhbGl6ZSBjb21tYW5kcyB3aG9zZSBLYWdnbGUgcnVudGltZSBkZWZhdWx0cyBxdWFudGl0eSB0byBvbmUuIiIiCgogICAg'
    'ZGVmIG5vcm1hbGl6ZV91bml0KGNvbW1hbmQpOgogICAgICAgIGlmIGlzaW5zdGFuY2UoY29tbWFuZCwgbGlzdCkgYW5kIGxlbihjb21tYW5kKSA9PSAy'
    'IGFuZCBjb21tYW5kWzBdIGluIHsiUElDS1VQIiwgIlBMQUNFIn06CiAgICAgICAgICAgIHJldHVybiBbKmNvbW1hbmQsIDFdCiAgICAgICAgcmV0dXJu'
    'IGNvbW1hbmQKCiAgICByZXR1cm4gewogICAgICAgICJmYXJtZXIiOiBub3JtYWxpemVfdW5pdChhY3Rpb24uZ2V0KCJmYXJtZXIiLCBbIlBBU1MiXSkp'
    'LAogICAgICAgICJoYW5kcyI6IFtub3JtYWxpemVfdW5pdChjb21tYW5kKSBmb3IgY29tbWFuZCBpbiBhY3Rpb24uZ2V0KCJoYW5kcyIsIFtdKV0sCiAg'
    'ICAgICAgIm1hcmtldCI6IGxpc3QoYWN0aW9uLmdldCgibWFya2V0IiwgW10pKSwKICAgIH0KCmRlZiBjbGVhbihhY3Rpb246IEFueSkgLT4gZGljdFtz'
    'dHIsIEFueV06CiAgICBpZiBub3QgaXNpbnN0YW5jZShhY3Rpb24sIGRpY3QpOgogICAgICAgIHJldHVybiB7ImZhcm1lciI6IFsiUEFTUyJdLCAiaGFu'
    'ZHMiOiBbXSwgIm1hcmtldCI6IFtdfQoKICAgIGRlZiB1bml0KGNvbW1hbmQ6IEFueSkgLT4gbGlzdFtBbnldOgogICAgICAgIGlmIG5vdCAoaXNpbnN0'
    'YW5jZShjb21tYW5kLCBsaXN0KSBhbmQgY29tbWFuZCBhbmQgaXNpbnN0YW5jZShjb21tYW5kWzBdLCBzdHIpKToKICAgICAgICAgICAgcmV0dXJuIFsi'
    'UEFTUyJdCiAgICAgICAgb3BlcmF0aW9uLCAqYXJndW1lbnRzID0gY29tbWFuZAogICAgICAgIGNvZXJjZWQ6IGxpc3RbQW55XSA9IFtdCiAgICAgICAg'
    'Zm9yIGFyZ3VtZW50IGluIGFyZ3VtZW50czoKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShhcmd1bWVudCwgc3RyKSBhbmQgYXJndW1lbnQubHN0cmlw'
    'KCItIikuaXNkaWdpdCgpOgogICAgICAgICAgICAgICAgY29lcmNlZC5hcHBlbmQoaW50KGFyZ3VtZW50KSkKICAgICAgICAgICAgZWxpZiBpc2luc3Rh'
    'bmNlKGFyZ3VtZW50LCAoc3RyLCBpbnQpKToKICAgICAgICAgICAgICAgIGNvZXJjZWQuYXBwZW5kKGFyZ3VtZW50KQogICAgICAgICAgICBlbHNlOgog'
    'ICAgICAgICAgICAgICAgcmV0dXJuIFsiUEFTUyJdCiAgICAgICAgaWYgb3BlcmF0aW9uIGluIE5PX0FSR19PUFM6CiAgICAgICAgICAgIHJldHVybiBb'
    'b3BlcmF0aW9uXQogICAgICAgIGlmIG9wZXJhdGlvbiA9PSAiUExBTlQiOgogICAgICAgICAgICByZXR1cm4gW29wZXJhdGlvbiwgY29lcmNlZFswXV0g'
    'aWYgY29lcmNlZCBhbmQgaXNpbnN0YW5jZShjb2VyY2VkWzBdLCBzdHIpIGVsc2UgWyJQQVNTIl0KICAgICAgICBpZiBvcGVyYXRpb24gaW4gKCJQSUNL'
    'VVAiLCAiUExBQ0UiKToKICAgICAgICAgICAgaWYgbm90IGNvZXJjZWQgb3Igbm90IGlzaW5zdGFuY2UoY29lcmNlZFswXSwgc3RyKToKICAgICAgICAg'
    'ICAgICAgIHJldHVybiBbIlBBU1MiXQogICAgICAgICAgICBjb3VudCA9IGNvZXJjZWRbMV0gaWYgbGVuKGNvZXJjZWQpID4gMSBhbmQgaXNpbnN0YW5j'
    'ZShjb2VyY2VkWzFdLCBpbnQpIGVsc2UgMQogICAgICAgICAgICByZXR1cm4gW29wZXJhdGlvbiwgY29lcmNlZFswXSwgY291bnRdCiAgICAgICAgcmV0'
    'dXJuIFsiUEFTUyJdCgogICAgb3JkZXJzID0gW10KICAgIGZvciBvcmRlciBpbiBhY3Rpb24uZ2V0KCJtYXJrZXQiKSBvciBbXToKICAgICAgICBpZiBu'
    'b3QgKGlzaW5zdGFuY2Uob3JkZXIsIGxpc3QpIGFuZCBvcmRlciBhbmQgaXNpbnN0YW5jZShvcmRlclswXSwgc3RyKSk6CiAgICAgICAgICAgIGNvbnRp'
    'bnVlCiAgICAgICAgaWYgb3JkZXJbMF0gaW4gKCJISVJFIiwgIkJVWV9MQU5EIik6CiAgICAgICAgICAgIG9yZGVycy5hcHBlbmQoW29yZGVyWzBdXSkK'
    'ICAgICAgICBlbGlmIG9yZGVyWzBdIGluIE1BUktFVF9PUFMgYW5kIGxlbihvcmRlcikgPj0gMyBhbmQgaXNpbnN0YW5jZShvcmRlclsxXSwgc3RyKToK'
    'ICAgICAgICAgICAgcXVhbnRpdHkgPSBvcmRlclsyXQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHF1YW50aXR5LCBzdHIpIGFuZCBxdWFudGl0eS5s'
    'c3RyaXAoIi0iKS5pc2RpZ2l0KCk6CiAgICAgICAgICAgICAgICBxdWFudGl0eSA9IGludChxdWFudGl0eSkKICAgICAgICAgICAgaWYgaXNpbnN0YW5j'
    'ZShxdWFudGl0eSwgaW50KSBhbmQgcXVhbnRpdHkgPiAwOgogICAgICAgICAgICAgICAgb3JkZXJzLmFwcGVuZChbb3JkZXJbMF0sIG9yZGVyWzFdLCBx'
    'dWFudGl0eV0pCiAgICByZXR1cm4gewogICAgICAgICJmYXJtZXIiOiB1bml0KGFjdGlvbi5nZXQoImZhcm1lciIpKSwKICAgICAgICAiaGFuZHMiOiBb'
    'dW5pdChjb21tYW5kKSBmb3IgY29tbWFuZCBpbiBhY3Rpb24uZ2V0KCJoYW5kcyIpIG9yIFtdXSwKICAgICAgICAibWFya2V0Ijogb3JkZXJzLAogICAg'
    'fQoKRklSU1RfU0hPUF9TVEVQID0gNzIKU0VDT05EX1NIT1BfU1RFUCA9IDE0NAoKX1BBVEhTOiBkaWN0W3N0ciwgc3RyXSA9IHt9Cl9UQVBFUzogZGlj'
    'dFtzdHIsIGxpc3RbQW55XV0gPSB7fQpfUElOTkVEOiBzZXRbc3RyXSA9IHNldCgpCl9DQUNIRV9MSU1JVCA9IDEyOAoKCmRlZiBsb2FkX3RhcGUocGF0'
    'aDogUGF0aCkgLT4gbGlzdFtBbnldOgogICAgcmF3ID0ganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSlbImFjdGlvbnMi'
    'XQogICAgcmV0dXJuIFtub3JtYWxpemVfbGVnYWN5X2FjdGlvbihjbGVhbihhY3Rpb24pKSBmb3IgYWN0aW9uIGluIHJhd10KCgpkZWYgX2luaXQocGF0'
    'aHM6IGRpY3Rbc3RyLCBzdHJdLCBwaW5uZWQ6IHR1cGxlW3N0ciwgLi4uXSA9ICgpKSAtPiBOb25lOgogICAgX1BBVEhTLnVwZGF0ZShwYXRocykKICAg'
    'IF9QSU5ORUQudXBkYXRlKHBpbm5lZCkKCgpkZWYgdGFwZShuYW1lOiBzdHIpIC0+IGxpc3RbQW55XToKICAgICIiIkxvYWQgb24gZmlyc3QgdXNlLiBK'
    'b2JzIGFyZSBjYW5kaWRhdGUtbWFqb3IsIHNvIGEgd29ya2VyIGhvbGRzIG9ubHkgYSBmZXcgYXQgb25jZS4iIiIKICAgIGNhY2hlZCA9IF9UQVBFUy5n'
    'ZXQobmFtZSkKICAgIGlmIGNhY2hlZCBpcyBOb25lOgogICAgICAgIGlmIGxlbihfVEFQRVMpID49IF9DQUNIRV9MSU1JVDoKICAgICAgICAgICAgZm9y'
    'IGtleSBpbiBbayBmb3IgayBpbiBfVEFQRVMgaWYgayBub3QgaW4gX1BJTk5FRF06CiAgICAgICAgICAgICAgICBkZWwgX1RBUEVTW2tleV0KICAgICAg'
    'ICBjYWNoZWQgPSBfVEFQRVNbbmFtZV0gPSBsb2FkX3RhcGUoUGF0aChfUEFUSFNbbmFtZV0pKQogICAgcmV0dXJuIGNhY2hlZAoKCmRlZiBwbGF5KGNh'
    'bmRpZGF0ZTogc3RyLCBvcHBvbmVudDogc3RyLCBzZWVkOiBpbnQsIHNlYXQ6IGludCkgLT4gZGljdFtzdHIsIEFueV06CiAgICBvdXJzLCB0aGVpcnMg'
    'PSB0YXBlKGNhbmRpZGF0ZSksIHRhcGUob3Bwb25lbnQpCiAgICBlbmdpbmUgPSBFbmdpbmUoeyJlcGlzb2RlU3RlcHMiOiA3MjAsICJzZWVkIjogc2Vl'
    'ZCwgIndlZWRTcGF3bkNoYW5jZSI6IDAuMDA1fSkKICAgIHNob3BzOiBsaXN0W2xpc3Rbc3RyXV0gPSBbW10sIFtdXQogICAgZm9yIHN0ZXAgaW4gcmFu'
    'Z2UoNzE5KToKICAgICAgICBpbmRleCA9IHN0ZXAgKyAxCiAgICAgICAgaWYgaW5kZXggPj0gbGVuKG91cnMpIG9yIGluZGV4ID49IGxlbih0aGVpcnMp'
    'OgogICAgICAgICAgICBicmVhawogICAgICAgIGlmIHN0ZXAgPT0gRklSU1RfU0hPUF9TVEVQOgogICAgICAgICAgICBzaG9wc1swXSA9IGxpc3QoZW5n'
    'aW5lLm9ic2VydmF0aW9uKHNlYXQpWyJ0b3duIl0uZ2V0KCJ1bmxvY2tlZF9zaG9wcyIsIFtdKSkKICAgICAgICBlbGlmIHN0ZXAgPT0gU0VDT05EX1NI'
    'T1BfU1RFUDoKICAgICAgICAgICAgc2hvcHNbMV0gPSBsaXN0KGVuZ2luZS5vYnNlcnZhdGlvbihzZWF0KVsidG93biJdLmdldCgidW5sb2NrZWRfc2hv'
    'cHMiLCBbXSkpCiAgICAgICAgYWN0aW9uczogbGlzdFtBbnldID0gW05vbmUsIE5vbmVdCiAgICAgICAgYWN0aW9uc1tzZWF0XSA9IGVuZ2luZS5zYW5p'
    'dGl6ZV9hY3Rpb24oc2VhdCwgb3Vyc1tpbmRleF0pCiAgICAgICAgYWN0aW9uc1sxIC0gc2VhdF0gPSBlbmdpbmUuc2FuaXRpemVfYWN0aW9uKDEgLSBz'
    'ZWF0LCB0aGVpcnNbaW5kZXhdKQogICAgICAgIGVuZ2luZS5zdGVwX3R5cGVkKHsic2NoZW1hX3ZlcnNpb24iOiAxLCAiYWN0aW9ucyI6IGFjdGlvbnN9'
    'KQogICAgbW9uZXkgPSBbZmFybVsibW9uZXkiXSBmb3IgZmFybSBpbiBlbmdpbmUuc3RhdGVbImZhcm1zIl1dCiAgICBtYXJnaW4gPSBtb25leVtzZWF0'
    'XSAtIG1vbmV5WzEgLSBzZWF0XQogICAgcmV0dXJuIHsKICAgICAgICAiY2FuZGlkYXRlIjogY2FuZGlkYXRlLCAib3Bwb25lbnQiOiBvcHBvbmVudCwg'
    'InNlZWQiOiBzZWVkLCAic2VhdCI6IHNlYXQsCiAgICAgICAgIm1vbmV5IjogbW9uZXlbc2VhdF0sICJvcHBvbmVudF9tb25leSI6IG1vbmV5WzEgLSBz'
    'ZWF0XSwgIm1hcmdpbiI6IG1hcmdpbiwKICAgICAgICAicG9pbnQiOiAxLjAgaWYgbWFyZ2luID4gMCBlbHNlIDAuNSBpZiBtYXJnaW4gPT0gMCBlbHNl'
    'IDAuMCwKICAgICAgICAic2hvcDEiOiBzaG9wc1swXVswXSBpZiBzaG9wc1swXSBlbHNlIE5vbmUsCiAgICAgICAgInNob3AyIjogc2hvcHNbMV1bMV0g'
    'aWYgbGVuKHNob3BzWzFdKSA+IDEgZWxzZSBOb25lLAogICAgfQoKCmRlZiBfd29yayhqb2I6IHR1cGxlW3N0ciwgc3RyLCBpbnQsIGludF0pIC0+IGRp'
    'Y3Rbc3RyLCBBbnldOgogICAgcmV0dXJuIHBsYXkoKmpvYikKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVu'
    'dFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jYW5kaWRhdGVzIiwgdHlwZT1QYXRoLCBuYXJncz0i'
    'KyIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW9wcG9uZW50cyIsIHR5cGU9UGF0aCwgbmFyZ3M9IisiLCByZXF1aXJl'
    'ZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZWVkLXN0YXJ0IiwgdHlwZT1pbnQsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRk'
    'X2FyZ3VtZW50KCItLXNlZWQtY291bnQiLCB0eXBlPWludCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0Iiwg'
    'dHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13b3JrZXJzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9bWF4KDEs'
    'IChvcy5jcHVfY291bnQoKSBvciAyKSAtIDIpKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICBkZWYga2V5KHBhdGg6IFBhdGgpIC0+'
    'IHN0cjoKICAgICAgICByZXR1cm4gZiJ7cGF0aC5wYXJlbnQucGFyZW50Lm5hbWV9L3twYXRoLnN0ZW19IgoKICAgIHBhdGhzID0ge2tleShwKTogc3Ry'
    'KHApIGZvciBwIGluIGxpc3QoYXJncy5jYW5kaWRhdGVzKSArIGxpc3QoYXJncy5vcHBvbmVudHMpfQogICAgY2FuZGlkYXRlcyA9IFtrZXkocCkgZm9y'
    'IHAgaW4gYXJncy5jYW5kaWRhdGVzXQogICAgb3Bwb25lbnRzID0gW2tleShwKSBmb3IgcCBpbiBhcmdzLm9wcG9uZW50c10KICAgIHNlZWRzID0gbGlz'
    'dChyYW5nZShhcmdzLnNlZWRfc3RhcnQsIGFyZ3Muc2VlZF9zdGFydCArIGFyZ3Muc2VlZF9jb3VudCkpCgogICAgam9icyA9IFsoYywgbywgcywgc2Vh'
    'dCkKICAgICAgICAgICAgZm9yIGMgaW4gY2FuZGlkYXRlcyBmb3IgbyBpbiBvcHBvbmVudHMgZm9yIHMgaW4gc2VlZHMgZm9yIHNlYXQgaW4gKDAsIDEp'
    'XQogICAgcHJpbnQoZiJ7bGVuKGNhbmRpZGF0ZXMpfSBjYW5kaWRhdGVzIHgge2xlbihvcHBvbmVudHMpfSBvcHBvbmVudHMgeCB7bGVuKHNlZWRzKX0g'
    'c2VlZHMgIgogICAgICAgICAgZiJ4IDIgc2VhdHMgPSB7bGVuKGpvYnMpfSBnYW1lcyBvbiB7YXJncy53b3JrZXJzfSB3b3JrZXJzIikKCiAgICAjIFJl'
    'c3VsdHMgYXJlIGFwcGVuZGVkIHBlciBnYW1lIHNvIGEga2lsbCBjb3N0cyBvbmx5IHRoZSBnYW1lcyBzdGlsbCBpbiBmbGlnaHQsIGFuZCBhCiAgICAj'
    'IHJlcnVuIHdpdGggdGhlIHNhbWUgYXJndW1lbnRzIHJlc3VtZXMgaW5zdGVhZCBvZiByZXBsYXlpbmcgd2hhdCBpcyBhbHJlYWR5IHJlY29yZGVkLgog'
    'ICAgYXJncy5vdXRwdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhcnRpYWwgPSBhcmdzLm91dHB1dC53aXRo'
    'X3N1ZmZpeCgiLnBhcnRpYWwuanNvbmwiKQogICAgcm93czogbGlzdFtkaWN0W3N0ciwgQW55XV0gPSBbXQogICAgZmluaXNoZWQ6IHNldFt0dXBsZVtz'
    'dHIsIHN0ciwgaW50LCBpbnRdXSA9IHNldCgpCiAgICBpZiBwYXJ0aWFsLmV4aXN0cygpOgogICAgICAgIGZvciBsaW5lIGluIHBhcnRpYWwucmVhZF90'
    'ZXh0KGVuY29kaW5nPSJ1dGYtOCIpLnNwbGl0bGluZXMoKToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcm93ID0ganNvbi5sb2Fkcyhs'
    'aW5lKQogICAgICAgICAgICBleGNlcHQganNvbi5KU09ORGVjb2RlRXJyb3I6CiAgICAgICAgICAgICAgICBjb250aW51ZSAgICAgICAgICAgICAgICAg'
    'ICAgICAjIGEgdG9ybiBmaW5hbCBsaW5lIGZyb20gYW4gaW50ZXJydXB0ZWQgcnVuCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHJvdykKICAgICAgICAg'
    'ICAgZmluaXNoZWQuYWRkKChyb3dbImNhbmRpZGF0ZSJdLCByb3dbIm9wcG9uZW50Il0sIHJvd1sic2VlZCJdLCByb3dbInNlYXQiXSkpCiAgICAgICAg'
    'cHJpbnQoZiJyZXN1bWluZzoge2xlbihyb3dzKX0gZ2FtZXMgYWxyZWFkeSByZWNvcmRlZCIpCiAgICBqb2JzID0gW2pvYiBmb3Igam9iIGluIGpvYnMg'
    'aWYgam9iIG5vdCBpbiBmaW5pc2hlZF0KICAgIHByaW50KGYiICB7bGVuKGpvYnMpfSBnYW1lcyBsZWZ0IHRvIHBsYXkiKQoKICAgIHdpdGggcGFydGlh'
    'bC5vcGVuKCJhIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgc2luaywgXAogICAgICAgICAgICBQcm9jZXNzUG9vbEV4ZWN1dG9yKGFyZ3Mud29ya2Vycywg'
    'aW5pdGlhbGl6ZXI9X2luaXQsIGluaXRhcmdzPShwYXRocywgdHVwbGUob3Bwb25lbnRzKSkpIGFzIHBvb2w6CiAgICAgICAgZm9yIGRvbmUsIHJvdyBp'
    'biBlbnVtZXJhdGUocG9vbC5tYXAoX3dvcmssIGpvYnMsIGNodW5rc2l6ZT00KSwgMSk6CiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHJvdykKICAgICAg'
    'ICAgICAgc2luay53cml0ZShqc29uLmR1bXBzKHJvdykgKyAiXG4iKQogICAgICAgICAgICBpZiBkb25lICUgMjAwID09IDAgb3IgZG9uZSA9PSBsZW4o'
    'am9icyk6CiAgICAgICAgICAgICAgICBzaW5rLmZsdXNoKCkKICAgICAgICAgICAgICAgIHByaW50KGYiICB7ZG9uZX0ve2xlbihqb2JzKX0iLCBmbHVz'
    'aD1UcnVlKQoKICAgIGFyZ3Mub3V0cHV0LndyaXRlX3RleHQoanNvbi5kdW1wcyhyb3dzKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwYXJ0'
    'aWFsLnVubGluayhtaXNzaW5nX29rPVRydWUpCiAgICBwcmludChmIndyb3RlIHtsZW4ocm93cyl9IHJvd3MgdG8ge2FyZ3Mub3V0cHV0fSIpCgoKaWYg'
    'X19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='
)
open('/kaggle/working/run_shop_router.py', 'wb').write(base64.b64decode(RUNNER_B64))
print('runner written,', len(base64.b64decode(RUNNER_B64)), 'bytes')


In [ ]:
# Identify tapes by the sha256 stored inside each file, never by a manifest key: the pool
# was tarred on macOS, so non-ASCII team names differ in Unicode normalisation between the
# manifest strings and the extracted filenames, and key lookups miss those tapes.
OPPONENT_HASHES = ['25f0696b8a6a8b2f5a2587345b8def5f56417347a416dfc0c6d78ce78387ed1b', '2fe169b9330b0de6c237098a4c43a2a7bfdfd477503e2940ede9eb38f60688f7', 'b832b28da13db8f0d4e030981bbb2ee7c25ae2b7ba4ad56aa77e41feb77768e2']

def tape_hash(path):
    with open(path, 'r', encoding='utf-8') as handle:
        head = handle.read(4096)
    match = re.search(r'"sha256":"([0-9a-f]{64})"', head)
    return match.group(1) if match else None

t0 = time.time()
by_hash = {}
for path in tape_files:
    digest = tape_hash(path)
    if digest:
        by_hash.setdefault(digest, path)
missing = [h for h in OPPONENT_HASHES if h not in by_hash]
assert not missing, f'opponents missing from pool: {missing}'

opponents = [by_hash[h] for h in OPPONENT_HASHES]
banned = set(OPPONENT_HASHES)
candidates = sorted(p for p in tape_files
                    if tape_hash(p) not in banned and tape_hash(p) is not None)
print(f'{len(candidates)} candidates, {len(opponents)} opponents, '
      f'indexed in {time.time() - t0:.0f}s')

In [ ]:
# Stage 1b (tie-break). Stage 1 left 211 candidates tied at point rate 0.8333 competing
# for the 23 stage-2 places the 17 candidates at 1.0000 do not fill, and the frozen
# document names no tie-break. This stage re-measures all 228 at >= 0.8333 on 30 unused
# seeds. See Amendment 1 in analysis/v95_shop_pair_router_20260830/PREREGISTRATION.md.
#
# The list is frozen at sha256 70813bd4a4038f7507ae39595f98415bc70ed6bf941846cb2019ac9b
# 6616733b. Keys are NFC-normalised on both sides for the reason cell 6 matches on
# sha256: the pool was tarred on macOS, so names like 'b\u1eedu_ph\u1ea1m_qu\u1ed1c'
# round-trip in NFD and would silently fail to match (7 tapes hit this).
import gzip
import unicodedata

SHORTLIST_B64 = (
'H4sIABczlWoC/6Wa/W7jOA7A32X/PmD1QUnUs9wuBLVRE08cO3DsdLJPv3Ru4ridWDJ9xXSKpvyZEkVRJOX//tHHcwrntq2DEsoK1OLPOP7ehetQh5DO'
'Uijj0UkM4ZJiL//4zyrGCgPaMRmltZP/Y8RaRiutLFOPUUYpph4DFgWX8dJCfmzxcjjGcIznuKviw97o0Zq8rlecU1Z6ZOvTwlipN3DSoTduA+eVdo49'
'P60MoCis225XXdomHNrP2O1CqJpUV/vqrU4P3dJKZ/JjrtPPUzt0cfzkgSlvseBmTbyFfeza2YBpyEYVqP6hArTUdrWwMzb/5Lf2bfZkgQXhLu3a0y9H'
'8gBWaBZgUed37hdACym1tyUN3dC8mIJYI29AoWQ831BQcAz5ezxgjMd6MipD3inpDUN+tqVWjZ/cWeN6eXUPtevHM8o7yXk+OK84z/eqEHp2p6rvbqGO'
'3bH95aZae2f5FIACy6eQ1jA7o1eU0RIVn6JpbaC8l4ZNWdCPg5BhDWutUlsoh1wKJApgjxCMpuOMaw0jhNXA1UWhhg5dPuXBWfYILR0r7FWep3zrqXnS'
'x6BAG2DbkOI1CM3W5QEN24aOIk4h+XtFeQeFmP+CQkGxn21DpLjhJdeG6DR9cSmvlNLsyOa1NpJteW/MBmt4J7Vmr5f3FNu466WFEMCObJT/oETH1qWE'
'EZ5NUfahDHuEMJYFbMqAYJ+wlNp5BM+mvBfInhcFevR5G6ZIJcQtjv+eCTGA5UKotM/H0PRWXYZwexZG6F0+pn3E7rRv20uaOZISLIRSLavyK7unOiqe'
'wuVQXWPo0484HchKmry2JdSiwYLHL6FOaVSbtFIyRdlbPhc4xOaSurbZv0im1iGa6m5j8uv2G+KoWi8MLHWxfk+XhwElxV3HItwYMBgEhQoq4/KW/lF1'
'rXpWKVTIZRUcY9W34WM4xlP1LBZtoQT/nRozfunyZ1Udz3V8T831dH0uvzL5BLIe3uPl2MSZBVR+Xepb8/PndGZIXdhLp3iJ5JehbXbPfU4els94TtX7'
'IaY69NXp7bGe5GemkF69wgx4C5KtzVBOgYatjdbWF7LUVxgN0RR6FC8wWi7trORq0xTI0AMfMwIKXZdTVdMBUKe2ufef7nMbfUobPmYpQQM+Rqm7sGxM'
'AWrcgFGSa/iYBmucZ2Pox74RF6McDbxmW5K2G4IQBS/p2xP9mY76Z9NXSM+EKLRN7cUlqAsf3dDs6im8U6oARnEhL1XhQDz1zXTkoLd6rbCS0hfmMBN2'
'xRV5CtNuRbVWeNaoXyHsZSkOTMJOOlnoOc2EESSolU9GKRystMbdmQsl0UyYzmjQK4UllXUlfx/qvgqHFHdhTCtT9/ArZ7xxZhuLwoIqxP4FdiyJQOMm'
'vWOJA4V0e5Glc04UGq6noYl1uFTN/jCd+4LShazCpjrWVR/fD+ncduk6tWNAq3zcew3OdzBLox0rC7VBoxOafGGDRifBFyLTEuihkLQugEbQIm7R6Chf'
'MRs0Iu1GhA0acYyAW4xDhbXDDQ4whhlXOJIWQKVKTdvXINUqdChsATW4wuXQAmhGd82CbZPCR9uFWNezqwqHRegfqvaEkFMXVmAhxJ3jtW4D1a9N1TyT'
'4qlzs6DqnKpje26ne5F7QyBLdFV83jLZqScnl6Sb/VuKXeqmVZICsKDhOzNLt1frofOzkEf9zswuiFfrceC9YTKIYEtju/QPeUq6jNN5P+tuz7YWpcZ5'
'VxmFD/GS9vFzKik1aJ3PZC6HuGs/+9C/SH7kOsQrMNIXkOEzNYHIT3KzfZp6OKgKm3uJNEYU0q4lksI0mC0k/QkLd/FL5KwRL1ikH+8nt8yTtrx2ym8i'
'LVURuGG0WlECX7jAvwwENf2BSuhuuMbqHPtq8la0Nr8ytHtS856El/DiTl+shRTo6YxfgqpjFZ7//X7py+LQly4f+6Mydmo2mlLqOxf3lrwkv8u/iFPa'
'4PKZ0ZB2u9tkYJwaRWvEUfhCH++LOCW/BlaLK2UQcL24fr4QsGLs807Okvgp9rFrQ5+aU3xWz1C4MidHfzvEa9gP537CKKCrvDNdj4fbrouPkkg6MlZ+'
'S38j5nfl63TMb6HXEfMSf92okKJ34cLgG+EBvecRxjnvGATNeszDGfOgQEmIYxEOS/ed34jxnRxk6VAUP6Tm6JjvqgXiZxXqYUpGgGJpPpzdPm7TQkhR'
'eCnq9qObmg2CPDYvPByrZnZHCIWO4Vd5RdkKMp4/e8NvlTytl+GMR4MsrO5X+ZnlV40HTOkdhW/yHgt3Vl/lDWjJsY+ljc+Rd9ZJxZBHY51l2NOPDsF4'
'/tgplevllaBEgyN//+bIj0VkYb6X4ZioMrjdL5wejUirXMkvXnDe0PGeX7+/BiGSq+gHRbuP/suv4ZhOsQ7H8cMPe6vvP99/fTq9OaEKb0783xqsf/rs'
'3/8CEM23Me4uAAA=')
wanted = {unicodedata.normalize('NFC', k)
          for k in json.loads(gzip.decompress(base64.b64decode(SHORTLIST_B64)))}

def key(path):
    parts = path.split('/')
    return unicodedata.normalize('NFC', f'{parts[-3]}/{parts[-1][:-5]}')

before = len(candidates)
candidates = [p for p in candidates if key(p) in wanted]
print(f'{before} in pool, {len(wanted)} on the shortlist, {len(candidates)} selected')
assert len(candidates) == len(wanted), (
    f'shortlist did not line up with the pool: {len(candidates)} != {len(wanted)}')


In [ ]:
# Stage 1b: the 228 shortlisted candidates vs the same opponent, 30 seeds, both seats.
# 228 x 30 x 2 = 13,680 games. Seeds 14150001-14150030 are disjoint from stage 1
# (14100001-14100006), stage 2 (14200001-14202000) and stage 3 (14300001-14302000).
workers = max(1, (os.cpu_count() or 4) - 1)
cmd = [sys.executable, '/kaggle/working/run_shop_router.py',
       '--candidates', *candidates, '--opponents', opponents[0],
       '--seed-start', '14150001', '--seed-count', '30',
       '--output', '/kaggle/working/stage1b.json', '--workers', str(workers)]
print(f'{len(candidates)} candidates on {workers} workers')
t0 = time.time()
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout[-4000:]); print(proc.stderr[-4000:])
print(f'elapsed {(time.time() - t0) / 60:.1f} min')
proc.check_returncode()


In [ ]:
# Selection rule, fixed in Amendment 1 before this ran: rank by stage-1b point rate
# alone (stage-1 scores are not pooled in -- the 228 were selected on them), ties broken
# by stage-1b mean margin, then by name ascending. Take the top 40.
import collections, statistics as st
rows = json.load(open('/kaggle/working/stage1b.json'))
grouped = collections.defaultdict(list)
for row in rows:
    grouped[row['candidate']].append(row)
table = [(sum(r['point'] for r in rs) / len(rs),
          st.mean(r['margin'] for r in rs), name) for name, rs in grouped.items()]
table.sort(key=lambda t: (-t[0], -t[1], t[2]))
print(f'{len(table)} candidates over {len(rows)} games\n')
for point, margin, name in table[:40]:
    print(f'  {point:.4f}  {margin:>+10,.0f}  {name.split("/")[-1]}')
print(f'\nrank 40 point rate {table[39][0]:.4f}, rank 41 {table[40][0]:.4f}')
